# 03 · Modelling

**Project FORESIGHT — NorthBay Living**

Features, the leakage guard, the backtest, and the comparison that decides what ships.

Run `python scripts/03_train_backtest.py` first — this notebook reads its artifacts rather
than retraining, which takes about ten minutes.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from foresight.config import get_settings
from foresight.eda import PALETTE, apply_house_style

warnings.filterwarnings("ignore", category=FutureWarning)
apply_house_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

settings = get_settings()
settings.processed_dir

In [ ]:
import json

from foresight.features import (
    FEATURE_SPEC,
    assert_no_leakage,
    build_base_features,
    build_supervised_frame,
    build_weekly_calendar,
)

panel = pd.read_parquet(settings.processed_dir / "weekly_panel.parquet")
calendar = pd.read_parquet(settings.processed_dir / "calendar.parquet")
weekly_calendar = build_weekly_calendar(calendar)
horizons = tuple(range(1, settings.horizon_weeks + 1))

metrics = json.loads((settings.artifacts_dir / "metrics.json").read_text())
predictions = pd.read_parquet(settings.artifacts_dir / "backtest_predictions.parquet")

print(f"{len(FEATURE_SPEC.all_features)} features, {len(predictions):,} backtested rows")

## 1. The leakage contract

Every feature is one of exactly two kinds:

1. **History-derived** — a function of the SKU's own past, evaluated at the origin and
   inclusive of it. The origin week has finished by forecast time, so its own value is known.
2. **Calendar-derived** — attributes of the *target* week taken from the calendar dimension.
   Legitimate because the calendar is published in advance: NorthBay set next quarter's
   promotion dates months ago.

The realised `promo_flag` from the sales ledger is deliberately **not** used for the target
week. That is an outcome, not a plan.

In [ ]:
pd.DataFrame(
    {
        "feature": FEATURE_SPEC.all_features,
        "kind": [
            "calendar (target week)" if name.startswith("target_")
            else "horizon" if name == "horizon"
            else "history (origin)"
            for name in FEATURE_SPEC.all_features
        ],
    }
).groupby("kind").size().to_frame("count")

## 2. Proving it, rather than asserting it

The guard rebuilds every feature against a copy of the panel whose future has been
overwritten with noise. If a feature value at or before the cutoff changes, it read the
future — there is nowhere else the difference could come from.

In [ ]:
assert_no_leakage(panel, weekly_calendar, horizons)
print("PASS — no feature responds to post-cutoff data")

A guard that only ever passes proves nothing, so it is also tested against a deliberate leak.
Here is the classic one: a centred rolling window.

In [ ]:
import foresight.features as F
from foresight.exceptions import LeakageError

original = F.build_base_features

def leaky(frame):
    out = original(frame)
    out["units_roll_mean_4"] = (
        frame.sort_values(["sku_id", "week_start"])
        .groupby("sku_id")["units"]
        .transform(lambda s: s.rolling(4, min_periods=1, center=True).mean())
        .to_numpy()
    )
    return out

F.build_base_features = leaky
try:
    assert_no_leakage(panel, weekly_calendar, horizons)
    print("FAIL — the guard missed a real leak")
except LeakageError as exc:
    print(f"CAUGHT — {str(exc)[:110]}")
finally:
    F.build_base_features = original

This guard runs inside `scripts/03_train_backtest.py` as a build gate. If it trips, no model
is trained and nothing is written.

## 3. The backtest

Rolling origin. At each stop the model is trained only on rows whose **target week** has
already been observed, then asked to forecast the following weeks.

Filtering on the *origin* week instead — the obvious-looking `week_start <= origin` — admits
rows whose targets lie beyond the split. That is the bug that inflates backtest scores across
the industry, and it has no visible symptom.

In [ ]:
summary = {
    key: metrics[key]
    for key in (
        "folds", "test_observations",
        "baseline_wape", "naive_wape", "gbm_wape", "ensemble_wape",
        "selected_model", "selected_wape", "selected_improvement_vs_baseline",
        "gbm_interval_coverage", "ensemble_interval_coverage",
        "gbm_bias_relative", "ensemble_bias_relative", "baseline_bias_relative",
    )
    if key in metrics
}
pd.Series(summary).to_frame("value")

## 4. Does it win in every fold, or only on average?

In [ ]:
folds = pd.DataFrame(metrics["per_fold"])
folds[["origin_week", "baseline_wape", "gbm_wape", "ensemble_wape"]].style.format(
    {"baseline_wape": "{:.3f}", "gbm_wape": "{:.3f}", "ensemble_wape": "{:.3f}"}
)

Every fold. A model that wins on average while losing in a third of periods is not something
an operations team can rely on week to week.

## 5. Error against the horizon

In [ ]:
by_horizon = pd.read_parquet(settings.artifacts_dir / "metrics_by_horizon.parquet")

axes = by_horizon.plot(
    x="horizon",
    y=[c for c in ("baseline_wape", "wape", "ensemble_wape") if c in by_horizon],
    kind="bar",
    figsize=(9, 3.2),
    color=[PALETTE["baseline"], PALETTE["harbour_700"], PALETTE["ember_600"]],
)
axes.set_ylabel("WAPE")
axes.set_title("Error by weeks ahead")
plt.show()

Error is essentially flat from horizon 1 to horizon 8. That is the payoff of **direct**
multi-horizon forecasting: a recursive model feeds its own predictions back in and would
fan out visibly across the horizon.

## 6. Where does the ensemble actually earn its keep?

In [ ]:
regime_path = settings.artifacts_dir / "metrics_by_regime.parquet"
if regime_path.exists():
    display(pd.read_parquet(regime_path))
    print(json.dumps(metrics.get("ensemble_weights", {}), indent=2))
else:
    print("Run scripts/03_train_backtest.py without --skip-ensemble to populate this.")

The intermittent row is the most informative result in the project: given a free choice
across the whole simplex, the optimiser put **zero** weight on the gradient-boosted model for
slow-moving lines and handed the entire forecast to the category seasonal profile.

That is not a tuning artefact. It is the model reporting that a global GBM trained on
absolute error has nothing useful to say about a product that sells nothing most weeks.

## 7. Interval calibration

In [ ]:
print(f"raw GBM interval coverage:  {metrics['gbm_interval_coverage']:.1%}")
if "ensemble_interval_coverage" in metrics:
    print(f"after conformal calibration: {metrics['ensemble_interval_coverage']:.1%}")
print(f"target:                      {metrics['interval_coverage_target']:.0%}")

The raw quantile models are calibrated individually by pinball loss, which guarantees nothing
about the **joint** coverage of the band. Left uncorrected, the interval claims 80% and
delivers ~71%.

That matters more than it looks. The risk layer converts the interval width directly into a
standard deviation and then into a stockout probability, so an interval 15% too narrow makes
every safety stock too small and every reorder recommendation too late — silently, for all
200 products.

---

Full write-ups: [`../reports/model_architecture.md`](../reports/model_architecture.md) and
[`../reports/model_comparison.md`](../reports/model_comparison.md).